# 7.1_geographic_enrichment

## Objectives

- Load territorial mobility datasets
- Load municipality coordinates
- Load annual municipality population datasets
- Create annual population tables
- Merge:
    - coordinates
    - autonomous communities
    - annual population
- Validates merges
- Exports geo-enriched datasets


## Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import unicodedata

## Paths

In [2]:
PROCESSED_PATH = Path("../data/processed")
INTERIM_PATH_GEO = Path("../data/interim/geo")
RAW_PATH = Path("../data/raw")

INTERIM_PATH_GEO.mkdir(
    parents=True,
    exist_ok=True
)


## Load territorial datasets

In [3]:
df_spanish_all_countries = pd.read_parquet(
    PROCESSED_PATH / "07_master_mobility.parquet"
)

df_catalans_all_countries = pd.read_parquet(
    PROCESSED_PATH / "07_catalonia_mobility.parquet"
)

df_spanish_outside_core_europe = pd.read_parquet(
    PROCESSED_PATH / "07_mobility_non_europe.parquet"
)

df_spanish_commercial_outside_europe = pd.read_parquet(
    PROCESSED_PATH / "07_mobility_commercial_destinations.parquet"
)

df_spanish_commercial_outside_europe_and_USA = pd.read_parquet(
    PROCESSED_PATH / "07_mobility_commercial_non_usa_eu.parquet"
)

## Load geographic coordinates

In [4]:
df_geo = pd.read_excel(
    RAW_PATH / "listado-longitud-latitud-municipios-espana.xlsx"
)

df_geo_country = pd.read_csv(
    RAW_PATH / "country_coordinates.csv"
)


## Inspect geographic data

In [5]:
display(df_geo.head())

display(df_geo.columns)

display(df_geo_country)


,Comunidad,Provincia,Población,Latitud,Longitud,Altitud
0,Andalucía,Almería,Abla,37.14114,-2.780104,871.16840
1,Andalucía,Almería,Abrucena,37.13305,-2.797098,976.93870
2,Andalucía,Almería,Adra,36.74807,-3.022522,10.97898
3,Andalucía,Almería,Albánchez,37.28710,-2.181163,481.31230
4,Andalucía,Almería,Alboloduy,37.03319,-2.621750,388.43460


Index(['Comunidad', 'Provincia', 'Población', 'Latitud', 'Longitud',
       'Altitud'],
      dtype='object')

,Country,Alpha-2 code,Alpha-3 code,Numeric code,Latitude (average),Longitude (average)
0,Afghanistan,"""AF""","""AFG""","""4""","""33""","""65"""
1,Åland Islands,"""AX""","""ALA""","""248""","""60.116667""","""19.9"""
2,Albania,"""AL""","""ALB""","""8""","""41""","""20"""
3,Algeria,"""DZ""","""DZA""","""12""","""28""","""3"""
4,American Samoa,"""AS""","""ASM""","""16""","""-14.3333""","""-170"""
...,...,...,...,...,...,...
257,Wallis and Futuna,"""WF""","""WLF""","""876""","""-13.3""","""-176.2"""
258,Western Sahara,"""EH""","""ESH""","""732""","""24.5""","""-13"""
259,Yemen,"""YE""","""YEM""","""887""","""15""","""48"""
260,Zambia,"""ZM""","""ZMB""","""894""","""-15""","""30"""


## Clean geographic data

In [6]:
df_geo = df_geo.rename(
    columns={
        "CODIGOINE": "municipality_code",
        "LATITUD_ETRS89": "latitude",
        "LONGITUD_ETRS89": "longitude",
    }
)


In [7]:
datasets = {
    "spanish_all_countries": df_spanish_all_countries,
    "catalans_all_countries": df_catalans_all_countries,
    "spanish_outside_core_europe": df_spanish_outside_core_europe,
    "spanish_commercial_outside_europe": df_spanish_commercial_outside_europe,
    "spanish_commercial_outside_europe_and_USA": df_spanish_commercial_outside_europe_and_USA
}


## Population files

In [8]:
population_files = {
    2019: "pobmun19.xlsx",
    2020: "pobmun20.xlsx",
    2021: "pobmun21.xlsx",
    2022: "pobmun22.xlsx",
    2023: "pobmun23.xlsx",
    2024: "pobmun24.xlsx",
    2025: "pobmun25.xlsx"
}


## Build annual population table

In [9]:
population_list = []

for year, file_name in population_files.items():

    df_population = pd.read_excel(
        RAW_PATH /
        "Ine_serie_historica_poblacion_municipio_19-25" /
        file_name,
        skiprows=1
    )

    # Clean column names
    df_population.columns = (
        df_population.columns
        .str.strip()
    )

    # Province code
    df_population["province_code"] = (
        df_population["CPRO"]
        .astype(int)
    )

    # Municipality code (INE format)
    df_population["city_code"] = (
        df_population["CPRO"]
        .astype(str)
        .str.zfill(2)
        +
        df_population["CMUN"]
        .astype(str)
        .str.zfill(3)
    )

    # Detect population column automatically
    population_column = [
        col for col in df_population.columns
        if "POB" in col.upper()
    ][0]

    # Rename columns
    df_population = df_population.rename(
        columns={
            "PROVINCIA": "province",
            "NOMBRE": "city",
            population_column: "population"
        }
    )

    # Add year
    df_population["pop_year"] = year

    # relevant columns
    df_population = df_population[
        [
            "province_code",
            "province",
            "city_code",
            "city",
            "population",
            "pop_year"
        ]
    ]

    population_list.append(df_population)


## Concatenate population tables

In [10]:
df_population = pd.concat(
    population_list,
    ignore_index=True
)


## Prepare datasets

In [11]:
datasets = {
    "spanish_all_countries": df_spanish_all_countries,
    "catalans_all_countries": df_catalans_all_countries,
    "spanish_outside_core_europe": df_spanish_outside_core_europe,
    "spanish_commercial_outside_europe": df_spanish_commercial_outside_europe,
    "spanish_commercial_outside_europe_and_USA": df_spanish_commercial_outside_europe_and_USA
}


### Text normalization function

In [12]:
def clean_text(text):
    if pd.isna(text):
        return np.nan
    text = (
        str(text)
        .lower()
        .strip()
    )
    text = unicodedata.normalize(
        "NFKD",
        text
    )
    text = (
        text
        .encode("ascii", "ignore")
        .decode("utf-8")
    )
    return text


# Convert city names: "Papiol (El)" - "papiol, el"
def ine_article_format(name):
    if pd.isna(name):
        return np.nan
    name = clean_text(name)
    replacements = {
        " (el)": ", el",
        " (la)": ", la",
        " (los)": ", los",
        " (las)": ", las",
        " (l')": ", l'"
    }

    for old, new in replacements.items():
        name = name.replace(old, new)

    return name


### Standardize geographic datasets

In [13]:
df_geo = df_geo.rename(
    columns={
        "Población": "city",
        "Latitud": "latitude",
        "Longitud": "longitude",
        "Provincia": "province"
    }
)

df_geo["city_clean"] = (
    df_geo["city"]
    .apply(ine_article_format)
)

df_geo["province_clean"] = (
    df_geo["province"]
    .apply(clean_text)
)


## Merge coordinates

### Merge municipality coordinates

In [14]:
datasets_geo = {}

for name, df in datasets.items():

    df["city_clean"] = (
        df["depart_city"]
        .apply(clean_text)
    )

    df["province_clean"] = (
        df["depart_province"]
        .apply(clean_text)
    )

    df = (
        df.merge(
            df_geo[
                [
                    "city_clean",
                    "province_clean",
                    "latitude",
                    "longitude"
                ]
            ],
            on=[
                "city_clean",
                "province_clean"
            ],
            how="left"
        )
    )

    df = df.drop(
        columns=[
            "city_clean",
            "province_clean"
        ]
    )

    datasets_geo[
        f"{name}_geo"
    ] = df

In [15]:
df_spanish_all_countries_geo = (
    datasets_geo["spanish_all_countries_geo"]
)

df_catalans_all_countries_geo = (
    datasets_geo["catalans_all_countries_geo"]
)

df_spanish_outside_core_europe_geo = (
    datasets_geo["spanish_outside_core_europe_geo"]
)

df_spanish_commercial_outside_europe_geo = (
    datasets_geo[
        "spanish_commercial_outside_europe_geo"
    ]
)

df_spanish_commercial_outside_europe_and_USA_geo = (
    datasets_geo[
        "spanish_commercial_outside_europe_and_USA_geo"
    ]
)

In [16]:
datasets_geo[
    "spanish_all_countries_geo"
][
    ["latitude", "longitude"]
].isna().mean()


latitude     0.004099
longitude    0.004099
dtype: float64

## Merge annual population

In [17]:
df_population["city_code"] = (
    df_population["city_code"]
    .astype(str)
    .str.zfill(5)
)

for name, df in datasets_geo.items():

    datasets_geo[name]["depart_city_code"] = (
        datasets_geo[name]["depart_city_code"]
        .astype(str)
        .str.zfill(5)
    )

    datasets_geo[name] = (
        datasets_geo[name].merge(
            df_population,
            left_on=[
                "depart_city_code",
                "year"
            ],
            right_on=[
                "city_code",
                "pop_year"
            ],
            how="left"
        )
    )


## Create outbound intensity metric

`tourists_per_1000_inhabitants` measures long-haul, agency-mediated outbound intensity per capita. Filters applied:
- include_in_analysis == True (excludes intra-European short-haul)
- agency_profile in ["medium", "high"] (excludes low agency-dependency destinations)  
- excludes Morocco (ferry proximity inflates border municipalities)
- excludes USA (structural outlier in volume)

In [18]:
for name, df in datasets_geo.items():

    # Filter to long-haul destinations only before aggregating
    # Excludes border-crossing noise from European/neighbouring countries & USA
    df_filtered = datasets_geo[name][
        (datasets_geo[name]["include_in_analysis"] == True) &
        (datasets_geo[name]["agency_profile"].isin(["medium", "high"])) &
        (datasets_geo[name]["destination_clean"] != "estados unidos de america") &
        (datasets_geo[name]["destination_clean"] != "marruecos") &
        (datasets_geo[name]["population"] >= 3000)  # no small municipalities
    ]

    # Step 1: aggregate total tourists per municipality per year
    city_year_total = (
        df_filtered
        .groupby(["depart_city_code", "year"], as_index=False)["total_tourists"]
        .sum()
        .rename(columns={"total_tourists": "annual_tourists_city"})
    )

    # Step 2: extract population
    city_year_pop = (
        datasets_geo[name][["depart_city_code", "year", "population"]]
        .drop_duplicates()
    )

    # Step 3: calculate metric
    city_year = city_year_total.merge(city_year_pop, on=["depart_city_code", "year"], how="left")
    city_year["tourists_per_1000_inhabitants"] = (
        city_year["annual_tourists_city"] / city_year["population"]
    ) * 1000

    # Step 4: merge back
    datasets_geo[name] = datasets_geo[name].merge(
        city_year[["depart_city_code", "year", "tourists_per_1000_inhabitants"]],
        on=["depart_city_code", "year"],
        how="left"
    )

In [19]:
df_check = datasets_geo["spanish_all_countries_geo"]

barcelona = df_check[
    (df_check["depart_city"] == "Barcelona") &
    (df_check["year"] == 2019)
][["depart_city", "year", "population", "tourists_per_1000_inhabitants"]].drop_duplicates()

display(barcelona)

,depart_city,year,population,tourists_per_1000_inhabitants
1120,Barcelona,2019,1636762.0,47.193789


In [20]:
df_check = datasets_geo["spanish_all_countries_geo"]

comparacio = df_check[
    df_check["year"] == 2025
][["depart_city", "depart_province", "population", "tourists_per_1000_inhabitants"]].drop_duplicates().sort_values("tourists_per_1000_inhabitants", ascending=False).head(20)

display(comparacio)

,depart_city,depart_province,population,tourists_per_1000_inhabitants
548107,Pozuelo de Alarcón,Madrid,89770.0,96.646987
543409,Barcelona,Barcelona,1731649.0,87.939299
547514,Boadilla del Monte,Madrid,66349.0,82.714133
547816,Madrid,Madrid,3506730.0,81.918197
548170,"Rozas de Madrid, Las",Madrid,99037.0,79.970112
544343,Sant Cugat del Vallès,Barcelona,97983.0,76.074421
547915,Majadahonda,Madrid,73625.0,68.495756
547372,Alcobendas,Madrid,123342.0,64.714371
543628,Castelldefels,Barcelona,70057.0,48.631828
548524,Benalmádena,Málaga,78338.0,45.341980


## Validation checks

### Validate geographic coverage

In [21]:
print(
    datasets_geo["spanish_all_countries_geo"][
        ["latitude", "longitude"]
    ]
    .isna()
    .mean()
)


latitude     0.004099
longitude    0.004099
dtype: float64


In [22]:
missing_geo = (
    datasets_geo["spanish_all_countries_geo"][
        datasets_geo["spanish_all_countries_geo"][
            "latitude"
        ].isna()
    ][
        [
            "depart_city",
            "depart_province"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "depart_province",
            "depart_city"
        ]
    )
)

print(
    f"Missing municipalities: "
    f"{len(missing_geo)}"
)

display(missing_geo.head())


Missing municipalities: 36


,depart_city,depart_province
306062,Alcosser,Alicante/Alacant
10375,Balanegra,Almería
18,Erriberabeitia,Araba/Álava
775,Guadiana,Badajoz
1037,"Castell, Es","Balears, Illes"


### Validate population table

In [23]:
print(df_population.info())

display(
    df_population.isna().sum()
)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56919 entries, 0 to 56918
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   province_code  56919 non-null  int32 
 1   province       56919 non-null  object
 2   city_code      56919 non-null  object
 3   city           56919 non-null  object
 4   population     56919 non-null  int64 
 5   pop_year       56919 non-null  int64 
dtypes: int32(1), int64(2), object(3)
memory usage: 2.4+ MB
None


province_code    0
province         0
city_code        0
city             0
population       0
pop_year         0
dtype: int64

## Destination geographic enrichment

### Add destination country coordinates for Streamlit flow maps


In [24]:
# Load country coordinates dataset

df_country_coordinates = pd.read_csv(
    RAW_PATH / "country_coordinates.csv"
)

display(df_country_coordinates.head())


,Country,Alpha-2 code,Alpha-3 code,Numeric code,Latitude (average),Longitude (average)
0,Afghanistan,"""AF""","""AFG""","""4""","""33""","""65"""
1,Åland Islands,"""AX""","""ALA""","""248""","""60.116667""","""19.9"""
2,Albania,"""AL""","""ALB""","""8""","""41""","""20"""
3,Algeria,"""DZ""","""DZA""","""12""","""28""","""3"""
4,American Samoa,"""AS""","""ASM""","""16""","""-14.3333""","""-170"""


In [25]:
# Clean coordinates dataset

df_country_coordinates.columns = (
    df_country_coordinates.columns
    .str.strip()
)

for column in [
    "Country",
    "Latitude (average)",
    "Longitude (average)"
]:

    df_country_coordinates[column] = (
        df_country_coordinates[column]
        .astype(str)
        .str.replace('"', "")
        .str.strip()
    )

df_country_coordinates[
    "Latitude (average)"
] = pd.to_numeric(
    df_country_coordinates[
        "Latitude (average)"
    ],
    errors="coerce"
)

df_country_coordinates[
    "Longitude (average)"
] = pd.to_numeric(
    df_country_coordinates[
        "Longitude (average)"
    ],
    errors="coerce"
)

display(df_country_coordinates.head())


,Country,Alpha-2 code,Alpha-3 code,Numeric code,Latitude (average),Longitude (average)
0,Afghanistan,"""AF""","""AFG""","""4""",33.000000,65.0
1,Åland Islands,"""AX""","""ALA""","""248""",60.116667,19.9
2,Albania,"""AL""","""ALB""","""8""",41.000000,20.0
3,Algeria,"""DZ""","""DZA""","""12""",28.000000,3.0
4,American Samoa,"""AS""","""ASM""","""16""",-14.333300,-170.0


In [26]:
# Destination mapping
# Manual destination mapping: Spanish -> English names for coordinates merge.
# Update this dictionary if new destinations are added to the pipeline.
# Source: INE destination_clean values mapped to country names in country_coordinates.csv

destination_mapping = {
    "emiratos arabes unidos":"United Arab Emirates",    
    "islandia":"Iceland",
    "turquia":"Turkey",
    "japon":"Japan",
    "china":"China",
    "colombia":"Colombia",
    "mexico":"Mexico",
    "cuba":"Cuba",
    "noruega":"Norway",
    "estados unidos de america":"United States",
    "tailandia":"Thailand",
    "finlandia":"Finland",
    "egipto":"Egypt",
    "republica dominicana":"Dominican Republic",
    "argentina":"Argentina",
    "india":"India",
    "georgia":"Georgia",
    "armenia":"Armenia",
    "senegal":"Senegal",
    "sudafrica": "South Africa",
    "kenia":"Kenya",
    "tunez":"Tunisia",
    "tanzania":"Tanzania, United Republic of",
    "costa rica" :"Costa Rica",
    "guatemala":"Guatemala",
    "panama":"Panama",
    "brasil":"Brazil",
    "peru":"Peru",
    "chile":"Chile",
    "indonesia":"Indonesia",
    "jordania":"Jordan",
    "camboya":"Cambodia",
    "australia":"Australia",
    "maldivas":"Maldives",
    "vietnam":"Vietnam",
    "cabo verde": "Cape Verd",
    "uganda":"Uganda",
    "sri lanka":"Sri Lanka",
    "uzbekistan":"Uzbekistan",
    "nepal":"Nepal",
    "seychelles":"Seychelles",
    "oman":"Oman",
    "azerbaiyan":"Azerbaijan",
    "kirguistan":"Kyrgyzstan",
    "canada": "Canada",
    "corea":  "South Korea"    
}


In [27]:
# Create English destination column

datasets_geo[
    "spanish_all_countries_geo"
][
    "destination_english"
] = (
    datasets_geo[
        "spanish_all_countries_geo"
    ][
        "destination_clean"
    ]
    .map(destination_mapping)
)

display(
    datasets_geo[
        "spanish_all_countries_geo"
    ][
        [
            "destination_clean",
            "destination_english"
        ]
    ]
    .drop_duplicates()
    .sort_values("destination_clean")
)


,destination_clean,destination_english
1120,albania,NaN
1,alemania,NaN
35,andorra,NaN
5328,angola,NaN
5366,arabia saudi,NaN
...,...,...
5389,uzbekistan,Uzbekistan
1187,venezuela,NaN
1207,vietnam,Vietnam
338182,yemen,NaN


In [28]:
# Merge destination coordinates

datasets_geo[
    "spanish_all_countries_geo"
] = (
    datasets_geo[
        "spanish_all_countries_geo"
    ]
    .merge(
        df_country_coordinates[
            [
                "Country",
                "Latitude (average)",
                "Longitude (average)"
            ]
        ],
        left_on="destination_english",
        right_on="Country",
        how="left"
    )
)


In [29]:
# Rename destination coordinates columns

datasets_geo[
    "spanish_all_countries_geo"
] = (
    datasets_geo[
        "spanish_all_countries_geo"
    ]
    .rename(
        columns={
            "Latitude (average)":"destination_lat",
            "Longitude (average)":"destination_lon"
        }
    )
)


In [30]:
# Validate missing destination coordinates

missing_destination_coordinates = (
    datasets_geo[
        "spanish_all_countries_geo"
    ][
        datasets_geo[
            "spanish_all_countries_geo"
        ][
            "destination_lat"
        ].isna()
    ][
        "destination_clean"
    ]
    .drop_duplicates()
    .sort_values()
)

display(missing_destination_coordinates)


1120           albania
1             alemania
35             andorra
5328            angola
5366      arabia saudi
              ...     
1146           ucrania
4001           uruguay
1187         venezuela
338182           yemen
1169          zimbabwe
Name: destination_clean, Length: 110, dtype: object

In [31]:
print(list(missing_destination_coordinates))

['albania', 'alemania', 'andorra', 'angola', 'arabia saudi', 'argelia', 'austria', 'bahamas', 'bahrein', 'bangladesh', 'barbados', 'belarus', 'belgica', 'benin', 'bhutan', 'bolivia', 'bosnia y herzegovina', 'botswana', 'bulgaria', 'burkina faso', 'cabo verde', 'camerun', 'chad', 'chipre', 'costa de marfil', 'croacia', 'dinamarca', 'djibouti', 'ecuador', 'el salvador', 'eslovenia', 'estonia', 'etiopia', 'fiji', 'filipinas', 'francia', 'gambia', 'ghana', 'gibraltar', 'grecia', 'guinea', 'guinea ecuatorial', 'guyana', 'honduras', 'hungria', 'iran', 'iraq', 'irlanda', 'israel', 'italia', 'jamaica', 'kazajstan', 'kuwait', 'laos', 'letonia', 'libano', 'libia', 'liechtenstein', 'lituania', 'luxemburgo', 'macedonia del norte', 'madagascar', 'malasia', 'mali', 'malta', 'marruecos', 'mauricio', 'mauritania', 'moldavia', 'monaco', 'mongolia', 'montenegro', 'mozambique', 'myanmar', 'namibia', 'nicaragua', 'niger', 'nigeria', 'nueva zelanda', 'otros paises de asia', 'otros paises de europa', 'paise

## Export geo-enriched datasets

In [32]:
for name, df in datasets_geo.items():

    df.to_parquet(
        INTERIM_PATH_GEO /
        f"07_{name}.parquet",
        index=False
    )

    print(
        f"Exported: 07_{name}.parquet"
    )


Exported: 07_spanish_all_countries_geo.parquet
Exported: 07_catalans_all_countries_geo.parquet
Exported: 07_spanish_outside_core_europe_geo.parquet
Exported: 07_spanish_commercial_outside_europe_geo.parquet
Exported: 07_spanish_commercial_outside_europe_and_USA_geo.parquet


## Final check

In [33]:
# Base dataframe

df_geo = datasets_geo[
    "spanish_all_countries_geo"
]

# Streamlit filtered dataframe

df_streamlit_geo = (
    df_geo[
        (
            df_geo[
                "include_in_analysis"
            ] == True
        )
        &
        (
            df_geo[
                "agency_profile"
            ].isin(
                ["medium", "high"]
            )
        )
    ]
    .copy()
)

# Check columns

display(
    df_streamlit_geo.columns.tolist()
)

# Check coordinates

display(
    df_streamlit_geo[
        [
            "destination_clean",
            "destination_lat",
            "destination_lon"
        ]
    ]
    .drop_duplicates()
    .sample(10)
)

# Export parquet

df_streamlit_geo.to_parquet(
    INTERIM_PATH_GEO /
    "07_streamlit_dataset.parquet",
    index=False
)

['period',
 'depart_city_code',
 'depart_city',
 'destination_code',
 'destination',
 'total_tourists',
 'depart_province_code',
 'depart_province',
 'year',
 'month',
 'destination_clean',
 'continent',
 'eu',
 'mediterranean',
 'flight_dependency',
 'agency_profile',
 'destination_segment',
 'include_in_analysis',
 'season',
 'covid_period',
 'autonomous_community',
 'latitude',
 'longitude',
 'province_code',
 'province',
 'city_code',
 'city',
 'population',
 'pop_year',
 'tourists_per_1000_inhabitants',
 'destination_english',
 'Country',
 'destination_lat',
 'destination_lon']

,destination_clean,destination_lat,destination_lon
50,colombia,4.0,-72.0
52,japon,36.0,138.0
5344,uganda,1.0,32.0
5385,sri lanka,7.0,81.0
1196,jordania,31.0,36.0
28431,oman,21.0,57.0
53,turquia,39.0,35.0
1173,costa rica,10.0,-84.0
943,egipto,27.0,30.0
1162,kenia,1.0,38.0
